# NWB Starter Notebook: Demo reversal learning electrophysiology dataset

**Dataset ID:** `DEMO_REVERSAL_EPHYS`  
**Source:** demo  
**URL:** [https://example.org/demo/reversal-ephys](https://example.org/demo/reversal-ephys)  
**Asset:** `data/seed/compiled/DEMO_REVERSAL_EPHYS.nwb`  
**Generated:** 2026-05-23 05:17 UTC

## Dataset Metadata

- **Species:** mouse
- **Modalities:** extracellular_ephys
- **Tasks:** reversal_learning
- **Behaviors:** choice, reward, omission, error
- **Brain Regions:** OFC, striatum

## Description

Mouse OFC and striatum extracellular electrophysiology during probabilistic reversal learning with choice, reward omission, reversal points, and trial outcomes.

## Dataset Card

**Summary:** Demo reversal learning electrophysiology dataset; matched to Reversal Learning; with extracellular_ephys.

**Analysis Readiness Score:** 95/100

**Strengths:**
- Uses NWB or BIDS metadata standards.
- Behavioral variables are present or inferable.
- Trial or event structure is available.

**Limitations:**
- Processed data availability is unclear.

**Suggested Analyses:**
- latent_state_modeling
- perseveration_analysis
- post_reversal_adaptation
- q_learning_modeling
- reward_prediction_error

---
## Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
notebook_template_id = 'reversal_learning_basic'
notebook_template_warnings = []

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)
%matplotlib inline


## NWB File Path

In [ ]:
# TODO: Update this path to point to your local NWB file
nwb_path = Path('data/seed/compiled/DEMO_REVERSAL_EPHYS.nwb')

# Verify file exists
if not nwb_path.exists():
    print(f'WARNING: File not found: {nwb_path}')
    print('Please update nwb_path to point to your local copy.')
else:
    print(f'NWB file: {nwb_path}')
    print(f'Size: {nwb_path.stat().st_size / 1e6:.1f} MB')


## Load NWB File

In [ ]:
io = NWBHDF5IO(str(nwb_path), mode='r')
nwbfile = io.read()
print(f'Loaded: {nwbfile.identifier}')
nwbfile


## Session Metadata

In [ ]:
session_metadata = {
    'identifier': nwbfile.identifier,
    'session_description': nwbfile.session_description,
    'session_start_time': str(nwbfile.session_start_time),
    'timestamps_reference_time': str(nwbfile.timestamps_reference_time) if nwbfile.timestamps_reference_time else None,
    'experimenter': nwbfile.experimenter,
    'lab': nwbfile.lab,
    'institution': nwbfile.institution,
    'experiment_description': nwbfile.experiment_description,
    'keywords': list(nwbfile.keywords) if nwbfile.keywords else None,
}

for key, value in session_metadata.items():
    if value:
        print(f'{key}: {value}')


## Subject Metadata

In [ ]:
if nwbfile.subject is not None:
    subject = nwbfile.subject
    subject_metadata = {
        'subject_id': subject.subject_id,
        'species': subject.species,
        'sex': subject.sex,
        'age': subject.age,
        'date_of_birth': str(subject.date_of_birth) if subject.date_of_birth else None,
        'strain': subject.strain,
        'genotype': subject.genotype,
        'description': subject.description,
    }
    for key, value in subject_metadata.items():
        if value:
            print(f'{key}: {value}')
else:
    print('No subject metadata found.')


## Acquisition Objects

In [ ]:
acquisition_keys = list(nwbfile.acquisition.keys())
print(f'Found {len(acquisition_keys)} acquisition objects:')

if acquisition_keys:
    acq_info = []
    for key in acquisition_keys:
        obj = nwbfile.acquisition[key]
        acq_info.append({
            'name': key,
            'type': type(obj).__name__,
            'description': getattr(obj, 'description', '')[:50] if hasattr(obj, 'description') else '',
        })
    display(pd.DataFrame(acq_info))
else:
    print('No acquisition objects found.')


## Processing Modules

In [ ]:
processing_keys = list(nwbfile.processing.keys())
print(f'Found {len(processing_keys)} processing modules:')

if processing_keys:
    proc_info = []
    for key in processing_keys:
        module = nwbfile.processing[key]
        data_interfaces = list(module.data_interfaces.keys())
        proc_info.append({
            'module': key,
            'description': module.description[:50] if module.description else '',
            'data_interfaces': ', '.join(data_interfaces[:3]) + ('...' if len(data_interfaces) > 3 else ''),
            'n_interfaces': len(data_interfaces),
        })
    display(pd.DataFrame(proc_info))
else:
    print('No processing modules found.')


## Units Table (Spike Data)

In [ ]:
if nwbfile.units is not None:
    units_df = nwbfile.units.to_dataframe()
    print(f'Found {len(units_df)} units with columns: {list(units_df.columns)}')
    display(units_df.head(10))
else:
    print('No units table found.')


## Trials Table

In [ ]:
if nwbfile.trials is not None:
    trials_df = nwbfile.trials.to_dataframe()
    print(f'Found {len(trials_df)} trials with columns: {list(trials_df.columns)}')
    display(trials_df.head(10))
else:
    print('No trials table found.')


## Intervals (Other Time Tables)

In [ ]:
interval_keys = list(nwbfile.intervals.keys()) if nwbfile.intervals else []
# Also check for trials in intervals if not in main trials
if 'trials' in interval_keys:
    interval_keys.remove('trials')  # Already shown above

print(f'Found {len(interval_keys)} interval tables:')

if interval_keys:
    for key in interval_keys:
        interval_table = nwbfile.intervals[key]
        interval_df = interval_table.to_dataframe()
        print(f'\n--- {key} ({len(interval_df)} rows) ---')
        print(f'Columns: {list(interval_df.columns)}')
        display(interval_df.head(5))
else:
    print('No additional interval tables found.')


## TimeSeries Summary Helper

In [ ]:
def summarize_timeseries(ts):
    """Summarize a TimeSeries object."""
    info = {
        'name': ts.name,
        'type': type(ts).__name__,
        'description': ts.description[:60] if ts.description else '',
        'unit': ts.unit if hasattr(ts, 'unit') else None,
        'rate': ts.rate if hasattr(ts, 'rate') and ts.rate else None,
    }
    if hasattr(ts, 'data') and ts.data is not None:
        data = ts.data
        info['shape'] = data.shape if hasattr(data, 'shape') else len(data)
        info['dtype'] = str(data.dtype) if hasattr(data, 'dtype') else type(data).__name__
    return info


def list_all_timeseries(nwbfile):
    """List all TimeSeries in the file."""
    all_ts = []
    
    # Acquisition
    for name, obj in nwbfile.acquisition.items():
        info = summarize_timeseries(obj)
        info['location'] = 'acquisition'
        all_ts.append(info)
    
    # Processing modules
    for mod_name, module in nwbfile.processing.items():
        for di_name, di in module.data_interfaces.items():
            if hasattr(di, 'data'):
                info = summarize_timeseries(di)
                info['location'] = f'processing/{mod_name}'
                all_ts.append(info)
    
    return pd.DataFrame(all_ts)


ts_summary = list_all_timeseries(nwbfile)
print(f'Found {len(ts_summary)} TimeSeries objects:')
display(ts_summary)


## Trial Column Analysis

In [ ]:
if nwbfile.trials is not None:
    trials_df = nwbfile.trials.to_dataframe()
    
    # Column summary
    col_summary = []
    for col in trials_df.columns:
        col_data = trials_df[col]
        col_summary.append({
            'column': col,
            'dtype': str(col_data.dtype),
            'missing': int(col_data.isna().sum()),
            'unique': col_data.nunique() if col_data.dtype != 'object' else min(col_data.nunique(), 20),
            'sample': str(col_data.dropna().iloc[0])[:30] if len(col_data.dropna()) > 0 else 'N/A',
        })
    
    print('Trial columns summary:')
    display(pd.DataFrame(col_summary))
else:
    print('No trials table available for analysis.')


## Basic Event Histogram

In [ ]:
if nwbfile.trials is not None:
    trials_df = nwbfile.trials.to_dataframe()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Trial duration histogram
    if {'start_time', 'stop_time'}.issubset(trials_df.columns):
        duration = trials_df['stop_time'] - trials_df['start_time']
        axes[0].hist(duration, bins=30, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Duration (s)')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Trial Duration Distribution')
        axes[0].axvline(duration.median(), color='red', linestyle='--', label=f'Median: {duration.median():.2f}s')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'No start_time/stop_time columns', ha='center', va='center')
        axes[0].set_title('Trial Duration')
    
    # Trial start times
    if 'start_time' in trials_df.columns:
        axes[1].hist(trials_df['start_time'], bins=30, edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('Time (s)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Trial Start Times')
    else:
        axes[1].text(0.5, 0.5, 'No start_time column', ha='center', va='center')
        axes[1].set_title('Trial Timing')
    
    plt.tight_layout()
    plt.show()
else:
    print('No trials table found for histogram.')


---
## Selected Notebook Template

## Notebook Template

**Template ID:** `reversal_learning_basic`

**Template:** Reversal learning notebook

Reversal learning analysis notebook with reward/omission, reversal, win-stay lose-shift, and Q-learning recipe cells.

---
## Selected Analysis Recipes

## Analysis Recipe: Reversal learning starter analyses

Reward/omission alignment, pre/post reversal analysis, win-stay lose-shift behavior, and Q-learning scaffolding.

**Recipe ID:** `reversal_learning_basic`

**Analyses:** reward_omission_aligned_activity, pre_post_reversal_analysis, win_stay_lose_shift, q_learning_modeling

**Expected fields:** trials, choice, reward, omission, reversal_point

## Recipe: Reversal learning starter analyses

Use reward/omission events and reversal points to inspect flexible updating before fitting a simple value model.

### Reward and omission aligned activity

In [ ]:
# Recipe: reward/omission aligned activity
if 'trials_df' in globals():
    event_cols = [c for c in trials_df.columns if any(k in c.lower() for k in ['reward', 'omission', 'feedback'])]
    print('Candidate reward/omission columns:', event_cols)
    # TODO: align neural features to reward and omission events.


### Pre/post reversal behavior

In [ ]:
# Recipe: pre/post reversal analysis
if 'trials_df' in globals():
    reversal_cols = [c for c in trials_df.columns if 'reversal' in c.lower() or 'block' in c.lower()]
    print('Candidate reversal/block columns:', reversal_cols)
    # TODO: compare choice accuracy and neural activity before vs after reversal points.


### Win-stay lose-shift scaffold

In [ ]:
# Recipe: win-stay lose-shift behavior
if 'trials_df' in globals():
    choice_cols = [c for c in trials_df.columns if 'choice' in c.lower()]
    reward_cols = [c for c in trials_df.columns if 'reward' in c.lower() or 'outcome' in c.lower()]
    print('Candidate choice columns:', choice_cols)
    print('Candidate reward/outcome columns:', reward_cols)
    # TODO: compute P(stay | previous rewarded) and P(shift | previous omission).


### Q-learning scaffold

In [ ]:
# Recipe: simple Q-learning scaffold
import numpy as np

def q_learning_update(q_values, choice, reward, alpha=0.2):
    prediction_error = reward - q_values[choice]
    q_values[choice] += alpha * prediction_error
    return q_values, prediction_error

print('Q-learning update helper ready. Map trial choices/rewards to integer arrays before fitting.')


---
## TODO: Analysis Roadmap

Below are suggested next steps for analysis. Uncomment and customize as needed.


In [ ]:
# =============================================================================
# TODO 1: Event Alignment
# =============================================================================
# Align neural data to behavioral events (e.g., stimulus onset, reward, choice)
#
# def align_to_event(spike_times, event_times, window=(-0.5, 1.0)):
#     """Align spike times to event times within a window."""
#     aligned = []
#     for event in event_times:
#         mask = (spike_times >= event + window[0]) & (spike_times <= event + window[1])
#         aligned.append(spike_times[mask] - event)
#     return aligned
#
# Example:
# event_times = trials_df['start_time'].values
# aligned_spikes = align_to_event(unit_spike_times, event_times)


In [ ]:
# =============================================================================
# TODO 2: Neural Decoding
# =============================================================================
# Decode behavioral variables from neural activity
#
# from sklearn.model_selection import cross_val_score
# from sklearn.linear_model import LogisticRegression
#
# def decode_choice(neural_features, choice_labels):
#     """Decode choice from neural features using logistic regression."""
#     clf = LogisticRegression(max_iter=1000)
#     scores = cross_val_score(clf, neural_features, choice_labels, cv=5)
#     return scores.mean(), scores.std()
#
# Example:
# accuracy, std = decode_choice(firing_rates, trials_df['choice'].values)
# print(f'Decoding accuracy: {accuracy:.2f} +/- {std:.2f}')


In [ ]:
# =============================================================================
# TODO 3: Behavioral Analysis
# =============================================================================
# Analyze behavioral patterns and performance
#
# def compute_psychometric(trials_df, stimulus_col, choice_col):
#     """Compute psychometric curve from trial data."""
#     grouped = trials_df.groupby(stimulus_col)[choice_col].mean()
#     return grouped
#
# def reaction_time_analysis(trials_df):
#     """Analyze reaction times by trial type."""
#     if 'reaction_time' in trials_df.columns:
#         return trials_df.groupby('trial_type')['reaction_time'].describe()
#
# Example:
# rt_stats = reaction_time_analysis(trials_df)


In [ ]:
# =============================================================================
# TODO 4: QC Report Export
# =============================================================================
# Generate a quality control report for the dataset
#
# def generate_qc_report(nwbfile, output_path):
#     """Generate QC report for NWB file."""
#     report = {
#         'identifier': nwbfile.identifier,
#         'n_trials': len(nwbfile.trials.to_dataframe()) if nwbfile.trials else 0,
#         'n_units': len(nwbfile.units.to_dataframe()) if nwbfile.units else 0,
#         'acquisition_objects': list(nwbfile.acquisition.keys()),
#         'processing_modules': list(nwbfile.processing.keys()),
#         'missing_metadata': [],
#     }
#     
#     # Check for missing metadata
#     if not nwbfile.subject:
#         report['missing_metadata'].append('subject')
#     if not nwbfile.experimenter:
#         report['missing_metadata'].append('experimenter')
#     
#     # Save report
#     import json
#     with open(output_path, 'w') as f:
#         json.dump(report, f, indent=2, default=str)
#     return report
#
# qc_report = generate_qc_report(nwbfile, 'qc_report.json')


In [ ]:
# Save an inspection summary JSON for this notebook run
import json
from pathlib import Path

inspection_summary = {
    'dataset_id': 'DEMO_REVERSAL_EPHYS',
    'template_id': 'reversal_learning_basic',
    'n_acquisition_objects': len(nwbfile.acquisition) if 'nwbfile' in globals() else None,
    'n_processing_modules': len(nwbfile.processing) if 'nwbfile' in globals() else None,
    'n_trials': len(nwbfile.trials.to_dataframe()) if 'nwbfile' in globals() and nwbfile.trials is not None else None,
    'n_units': len(nwbfile.units.to_dataframe()) if 'nwbfile' in globals() and nwbfile.units is not None else None,
    'template_warnings': notebook_template_warnings if 'notebook_template_warnings' in globals() else [],
}
summary_path = Path(f'{inspection_summary["dataset_id"]}_inspection_summary.json')
summary_path.write_text(json.dumps(inspection_summary, indent=2, default=str))
print(f'Inspection summary written to {summary_path}')
inspection_summary


---
## Cleanup

In [ ]:
# Close the NWB file when done
io.close()
print('NWB file closed.')
